# Classification of fetal organs in US

Исследовательская классификация изображений. Архитектура исходного эксперимента сохранена.
Код пересмотрен: train/validation/test разделяются по пациентам, проверяются повторяющиеся изображения,
валидация не аугментируется. Результаты старого запуска доступны в истории Git; они не относятся к этому коду.
Новые метрики появятся только после запуска с данными. См. README и ../METHODOLOGY.md.

Перед запуском задайте DATA_ROOT (изображения), MANIFEST_PATH (CSV) и OUTPUT_DIR (локальные результаты).
Схема CSV: filename,label,patient_id[,split]. Снимки и манифест не публикуйте.


In [ ]:
# Запускайте из корня клонированного репозитория или из папки проекта.
from pathlib import Path
import sys
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'cv_common').is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри полного клона My_projects.')
sys.path.insert(0, str(REPO_ROOT))

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from cv_common.data import load_manifest, validate_images, patient_split, keras_generators, evaluate_classifier

from cv_common.data import read_rgb, create_run_directory, save_split_manifest

SEED = 12345
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
DATA_ROOT = Path(os.environ.get('DATA_ROOT', 'data')).resolve()
OUTPUT_DIR = create_run_directory(Path(os.environ.get('OUTPUT_DIR', REPO_ROOT / 'artifacts' / 'fetal-ultrasound')))


In [ ]:
CLASSES = ['Fetal abdomen', 'Fetal brain', 'Fetal femur', 'Fetal thorax']

In [ ]:
# Prefer verified patient metadata and official splits.
manifest_path = Path(os.environ.get('MANIFEST_PATH', DATA_ROOT / 'manifest.csv'))
if manifest_path.is_file() or os.environ.get('MANIFEST_PATH'):
    parts = load_manifest(manifest_path, DATA_ROOT, SEED)
else:
    if os.environ.get('ALLOW_FILENAME_PATIENT_IDS') != '1':
        raise ValueError('Provide manifest.csv or verify PatientNNNNN semantics and set ALLOW_FILENAME_PATIENT_IDS=1.')
    df = pd.read_csv(DATA_ROOT / 'image_label.csv', dtype=str)
    df = df.rename(columns={'Image_name': 'filename', 'Plane': 'label'})
    df['patient_id'] = df['filename'].str.extract(r'^(Patient\d+)_', expand=False)
    if df['patient_id'].isna().any():
        raise ValueError('Не удалось определить пациента; подготовьте проверенный манифест.')
    df['filename'] = df['filename'].map(lambda x: 'images/' + (x if x.endswith('.png') else x + '.png'))
    parts = patient_split(validate_images(df, DATA_ROOT), SEED)
if set().union(*(set(f['label']) for f in parts.values())) != set(CLASSES):
    raise ValueError('Метки манифеста не совпадают с ожидаемыми классами.')
# Сохраняем реальное разбиение локально; оно может содержать идентификаторы пациентов.
save_split_manifest(parts, DATA_ROOT, OUTPUT_DIR / 'split_manifest.csv')
print(pd.concat(parts.values()).groupby(['split', 'label']).size().unstack(fill_value=0))

In [ ]:
generators = keras_generators(parts, CLASSES, batch_size=16, vgg=True, augment={'rotation_range': 10, 'horizontal_flip': True}, seed=SEED)
train_generator = generators['train']
validation_generator = generators['validation']
(OUTPUT_DIR / 'classes.json').write_text(json.dumps(train_generator.class_indices, indent=2))

In [ ]:
# Просмотр исходного RGB; VGG16-преобразованные массивы не подходят для imshow.
fig, axes = plt.subplots(1, min(4, len(parts['train'])), figsize=(12, 3), squeeze=False)
for ax, path in zip(axes.ravel(), parts['train']['filename'].head(4)):
    with Image.open(path) as image:
        ax.imshow(image.convert('RGB'))
    ax.axis('off')
plt.show()

In [ ]:
# Воспользуемся предобученной моделью VGG16
from tensorflow.keras.applications import VGG16

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224) + (3,))

for layer in base_model.layers:
    layer.trainable = False
model = Sequential()
model.add(base_model)
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dense(4, activation='softmax'))

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
# Один критерий выбора модели; сохраняем лучшую эпоху по validation loss.
callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ModelCheckpoint(OUTPUT_DIR / 'best_model.keras', monitor='val_loss', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=.2, patience=2, min_lr=1e-6),
]
# Не задаём floor(n / batch_size): последний неполный батч тоже учитывается.
history = model.fit(train_generator, validation_data=validation_generator, epochs=100, callbacks=callbacks)
model = tf.keras.models.load_model(OUTPUT_DIR / 'best_model.keras')
(OUTPUT_DIR / 'history.json').write_text(json.dumps(history.history, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    epochs = np.arange(1, len(history.history[metric]) + 1)
    ax.plot(epochs, history.history[metric], label='train')
    ax.plot(epochs, history.history['val_' + metric], label='validation')
    ax.set(xlabel='Epoch', ylabel=metric)
    ax.legend()
plt.show()

## Независимый тест

Запускайте после фиксации архитектуры и гиперпараметров. Не используйте эти результаты
для дальнейшего выбора модели. Наличие patient_id не доказывает правильность его происхождения;
проверьте источники, повторные исследования и близкие дубли отдельно.

In [ ]:
report = evaluate_classifier(model, generators['test'])
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(pd.DataFrame(report['classification_report']).T)
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(report['confusion_matrix'])
ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
       xticklabels=CLASSES, yticklabels=CLASSES, xlabel='Predicted', ylabel='True')
plt.setp(ax.get_xticklabels(), rotation=90)
plt.show()

In [ ]:
# Пример инференса по локальному файлу. Предобработка совпадает с обучением.
def classify_image(image_path):
    image = read_rgb(image_path)
    array = np.array(image.resize((224, 224), Image.Resampling.NEAREST), dtype=np.float32)
    array = tf.keras.applications.vgg16.preprocess_input(array)
    probabilities = model.predict(array[None], verbose=0)[0]
    return dict(zip(CLASSES, probabilities.tolist()))